# Fine-tune the prompt-attack classifier on a Colab GPU

Runtime → Change runtime type → **T4 GPU**. This runs the same `classifier/train.py` as the local CPU runs, so metrics land in `results/experiments.jsonl` in the same format.

The decision threshold is chosen on the validation split, never the test split.

In [ ]:
!git clone -b finetuned-classifier https://github.com/aadraxe/LLM-middleware-security.git
%cd LLM-middleware-security
!pip install -q -r requirements.txt -r requirements-classifier.txt

## 1. Build the dataset
Downloads five Hugging Face datasets (licenses in `data/DATASET.md`) and writes seeded splits. Check the printed `sha256` values against `data/processed/summary.json` in the repo to confirm you got the same splits.

In [ ]:
!python data/build_dataset.py | tail -20

## 2. Train
Same settings as the local CPU DistilBERT run (2 epochs, 256 tokens) so the three models compare fairly. Results go to `results/colab/` so they never overwrite your local `results/experiments.jsonl`.

Check the GPU first: the first line should say `device=cuda`. If it says `cpu`, set Runtime → Change runtime type → T4 GPU.

In [ ]:
!python classifier/train.py --model microsoft/deberta-v3-small --run-name deberta-v3-small --epochs 2 --results-dir results/colab
!python classifier/train.py --model microsoft/deberta-v3-base --lora --run-name deberta-v3-base-lora --epochs 2 --results-dir results/colab
# Optional sanity check: same DistilBERT recipe on GPU (should roughly match the local CPU run)
# !python classifier/train.py --model distilbert-base-uncased --run-name distilbert-gpu --epochs 2 --results-dir results/colab

## 3. Compare runs

In [ ]:
import json
for line in open('results/colab/experiments.jsonl'):
    r = json.loads(line)
    print(f"{r['run_name']:28} thr={r['threshold']:.2f} test_f1={r['test']['f1']:.3f} bench_f1={r['benchmark_200']['f1']:.3f} bench_recall={r['benchmark_200']['recall']:.3f} bench_fpr={r['benchmark_200']['fpr']:.3f}")

## 4. Download the models and metrics
Unzip `colab_results.zip` in the repo root locally. It contains `models/deberta-v3-small/`, `models/deberta-v3-base-lora/` and `results/colab/`, and touches nothing else.

In [ ]:
!zip -qr colab_results.zip models/deberta-v3-small models/deberta-v3-base-lora results/colab
from google.colab import files
files.download('colab_results.zip')